# Module 2: Data Preprocessing & Machine Learning Fundamentals

## Overview

This module covers the essential **data preparation and modeling workflow** that transforms raw data into actionable machine learning models. You'll learn the systematic process data scientists follow, from initial data exploration to model evaluation.

**Learning Objectives:**
1. Understand the complete ML workflow and its iterative nature
2. Master exploratory data analysis techniques for quick insights
3. Apply data cleaning strategies (missing values, outliers)
4. Create meaningful features through engineering
5. Select optimal feature subsets to prevent overfitting
6. Evaluate models using appropriate metrics
7. Implement and optimize the k-Nearest Neighbors algorithm

**Why This Matters:** Data preprocessing typically consumes 60-80% of a data scientist's time. Mastering these fundamentals ensures your models learn from clean, informative data rather than noise.

---

# Part I: The Machine Learning Workflow

## 1.1 The Complete ML Lifecycle

Machine learning projects follow a structured, **iterative process** with 7 interconnected stages:

### The 7-Stage ML Workflow

**1. 🎯 Business Understanding**  
↓ *Define the problem and success metrics*

**2. 📊 Data Collection**  
↓ *Gather relevant data sources*

**3. 🔍 Exploratory Data Analysis**  
↓ *Investigate patterns and data quality*

**4. 🧹 Data Preparation**  
↓ *Clean, transform, and engineer features*

**5. 🤖 Modeling**  
↓ *Train ML algorithms*

**6. 📈 Model Evaluation**  
↓ *Assess performance and validity*

**7. 🚀 Model Deployment**  
→ *Back to step 1 (continuous improvement)*

---

### Understanding the Cycle

**Critical Insight:** This is a **cycle, not a waterfall**. After deployment, you'll often return to earlier stages based on:
- 🔄 New business requirements
- 📉 Model performance degradation  
- 📥 Availability of new data sources
- 👥 Changing user behavior patterns

**Real-World Example:** A deployed recommendation system might need retraining (back to steps 4-5) when user preferences shift seasonally, or require new features (back to steps 3-4) when competitors introduce new product categories.

> **🎓 Module Focus:** We'll dive deep into **stages 3-4 (EDA and Data Preparation)**, which form the foundation for successful modeling.

---

# Part II: Exploratory Data Analysis (EDA)

## 2.1 What is EDA?

**Exploratory Data Analysis** is the detective work of data science—investigating your dataset to uncover patterns, spot anomalies, and understand relationships **before** building models.

**Goal:** Answer critical questions like:
- What does my data look like?
- Are there missing values or errors?
- What's the distribution of each variable?
- Which features might be useful for prediction?

### The 4-Step EDA Framework

Follow this systematic approach for **rapid yet thorough** data understanding:

In [ ]:
import pandas as pd

# Load data
df = pd.read_excel("file.xlsx")

# Step 1: Quick look at the data
df.head()  # First 5 rows to understand structure

In [ ]:
# Step 2: Numerical analysis
df.describe()  
# Find: missing values, outliers, range, mean, std, quartiles

In [ ]:
# Step 3: Categorical analysis
df.describe(include='O')  
# Find: unique values, most frequent, useless columns (unique=1)

In [ ]:
# Step 4: Data cleaning decisions
df.drop(['useless_col'], axis=1, inplace=True)  # Remove irrelevant columns
# Other actions: rename columns, merge categories, handle dtypes

---

# Part III: Data Cleaning & Preprocessing

Now that we've explored the data, we need to **clean and prepare** it for modeling. Two critical challenges emerge: **missing values** and **outliers**.

## 3.1 Handling Missing Values

Missing data is ubiquitous in real-world datasets. Simply removing rows with missing values can lead to significant data loss. Instead, we use **imputation strategies** that intelligently fill gaps.

### Strategy 1: Forward Fill (Temporal Imputation)

**Purpose:** Use the most recent known value from the same entity (e.g., product, customer)

**When to Use:** The entity has historical values to reference

**How It Works:** Propagate the last observed value forward

**Example: SKU1009 (same product across weeks)**

| Week | Original Value | After Forward Fill |
|------|----------------|-------------------|
| Week 1 | Rank 5 | Rank 5 ✓ |
| Week 2 | Rank 6 | Rank 6 ✓ |
| Week 3 | NaN | Rank 6 (from Week 2) ✓ |
| Week 4 | Rank 7 | Rank 7 ✓ |
| Week 5 | NaN | Rank 7 (from Week 4) ✓ |
| Week 6 | Rank 8 | Rank 8 ✓ |

**⚠️ Limitation:** Doesn't work if ALL values are missing (no starting point to fill from)

### Strategy 2: Segment Median Imputation (Category-Based)

**Purpose:** Use typical values from similar entities when no historical data exists

**When to Use:** Entity has NO previous values (all NaN), or after forward fill still has gaps

**How It Works:** Fill with the median value from the entity's segment/category

**Problem: SKU1234 (new product, never ranked)**
- Week 1-104: All NaN
- → Forward fill cannot help (nothing to fill from!)

**Solution: Calculate segment median**

| SKU | Segment | Average Rank |
|-----|---------|--------------|
| SKU1009 | Electronics | 5 |
| SKU2000 | Electronics | 10 |
| SKU3000 | Electronics | **45** ← Median |
| SKU4000 | Electronics | 50 |
| SKU5000 | Electronics | 90 |

**Result:** SKU1234 (Electronics) → Fill all weeks with 45 ✓

---

### Best Practice: Combined Strategy

**📋 Recommendation:** Apply a **two-stage approach**:

1. **First:** 🔄 Forward fill (captures entity-specific trends)
2. **Then:** 📊 Segment median for remaining gaps (uses domain knowledge)

This hybrid method balances individual patterns with group-level insights.

---

## 3.2 Outlier Detection & Treatment

After handling missing values, the next challenge is **extreme values** that don't fit typical patterns.

### Understanding Outliers

**Definition:** Data points significantly different from the majority of observations—unusually high or low values

**Impact on ML Models:**
- Distort statistical measures (mean, variance, correlation)
- Skew model training toward extreme cases
- Reduce model generalization ability

**Important Question:** Are outliers **errors** (data quality issues) or **insights** (rare but valid events)? This context determines treatment strategy.

### Detection Methods

#### A. Visual Methods
- **Box Plot:** Shows quartiles and flags points beyond whiskers
- **Histogram:** Reveals distribution shape and extreme values
- **Scatter Plot:** Identifies unusual combinations of features

#### B. Statistical Methods
- **Z-Score:** Measures standard deviations from mean (typically |z| > 3 indicates outlier)
- **IQR (Interquartile Range):** Flags points below Q1 - 1.5×IQR or above Q3 + 1.5×IQR

### Treatment Methods

| Method | Description | Pros | Cons |
|--------|-------------|------|------|
| **1. Removal** | Delete outlier rows | • Clean dataset | • Data loss<br>• May remove valuable info<br>• Reduces sample size |
| **2. Transformation** | Apply mathematical functions (log, sqrt, Box-Cox) | • Reduces extreme influence<br>• Maintains relationships | • Changes data distribution<br>• Interpretation complexity |
| **3. Capping/Flooring** | Replace outliers with threshold values (e.g., 5th/95th percentile) | • Retains all rows<br>• Reduces extreme values<br>• Maintains sample size | • Arbitrary thresholds<br>• Can create artificial clusters |
| **4. Imputation** | Replace with mean/median | • Simple to implement<br>• Keeps all data | • Reduces variance<br>• Can distort distribution |

### When to Treat Outliers?

**Important:** Outlier treatment is NOT always necessary. Consider:

- **Domain Knowledge:** Is this value realistically possible?
- **Data Validity:** Is it a measurement error or genuine extreme case?
- **Sample Size:** Large datasets can tolerate outliers better
- **Business Context:** Might the outlier represent an important edge case?
- **Model Sensitivity:** Tree-based models are robust to outliers; linear models are not

---

# Part IV: Feature Engineering & Selection

With clean data in hand, we now focus on **creating and selecting the right features**—often the most impactful step in the ML workflow.

## 4.1 Feature Engineering: Creating Better Inputs

### The Power of Feature Engineering

**Definition:** Transforming raw data into features that better represent the underlying problem, enabling models to learn more effectively

**Core Principle:** `Raw Data → Better Features → Better Predictions`

**Why It Matters:** A simple model with great features often outperforms a complex model with poor features. Feature engineering encodes **domain knowledge** into your data.

### Example: Date Feature Engineering

Consider a raw date field—what can we extract from it?

**📅 Raw Data:**
- Date = `"2021-12-25"`
- ❌ Not directly useful for ML models

**⚙️ Engineered Features:**
- `Month` = 12
- `Is_December` = 1
- `Is_Holiday` = 1
- `Quarter` = 4
- `Days_to_year_end` = 6
- `Day_of_week` = 6 (Saturday)
- `Is_weekend` = 1

**🎯 Model Learns:**
- "December → high sales"
- "Holidays → high sales"
- "Weekends → different patterns"

### Common Feature Engineering Techniques

1. **Temporal Features:** Extract components from dates/times
2. **Aggregations:** Sum, mean, count over groups
3. **Ratios:** Create meaningful proportions (e.g., conversion_rate = purchases / visits)
4. **Binning:** Convert continuous variables to categories
5. **Encoding:** Transform categorical variables to numeric (one-hot, label encoding)
6. **Interaction Features:** Combine multiple features (e.g., price × quantity)

---

## 6. Feature Selection

### Definition

**Feature Selection:** Choosing which features to keep or remove from the dataset

**Goal:** Retain only useful features; eliminate redundant, irrelevant, or noisy ones

**Action:** Filter/remove features (opposite of feature engineering)

### The Goldilocks Problem: Finding the Right Number of Features

#### Problem 1: Overfitting (Too Many Features)

**Definition:** Model learns training data TOO well, including noise and random patterns

**Symptoms:**
- ✅ High accuracy on training data (e.g., 99%)
- ❌ Poor accuracy on test/real-world data (e.g., 60%)

**Causes:**
- Too many features (including irrelevant/noisy ones)
- Model memorizes training data instead of generalizing
- Captures random noise as real patterns

**Solution:** Remove irrelevant features, use regularization, or simplify model

#### Problem 2: Underfitting (Too Few Features)

**Definition:** Model is TOO simple and cannot learn even basic patterns

**Symptoms:**
- ❌ Poor accuracy on training data (e.g., 50%)
- ❌ Poor accuracy on test/real-world data (e.g., 48%)

**Causes:**
- Too few features (removed important predictors)
- Model lacks sufficient information
- Missing critical variables

**Solution:** Add more relevant features, increase model complexity

### Integrating Features into the ML Workflow

Here's how feature work fits into the broader process:

**1. 📥 RAW DATA**  
↓

**2. 🧹 DATA CLEANING**  
*Handle missing values, outliers*  
↓

**3. ⚙️ FEATURE ENGINEERING**  
*Create new features from existing data*  
↓

**4. ✂️ FEATURE SELECTION**  
*Remove bad/redundant features*  
↓

**5. ✅ FINAL DATASET FOR MODELING**  
*Optimal, clean feature set*

---

**💡 Practical Tip:** Start with many features (be creative in engineering), then systematically remove poor performers through selection. It's easier to remove features than to realize later you're missing critical ones.

---

## 7. k-Nearest Neighbors (kNN) Algorithm

### Overview

**Type:** Instance-based, non-parametric, lazy learning algorithm

**Key Characteristic:** No training phase—simply stores all training data and makes predictions based on similarity

**Use Cases:**
- Classification (predict categories)
- Regression (predict continuous values)

### How kNN Works


Step 1: New data point arrives  

Step 2: Calculate distance to ALL training points  

Step 3: Find k CLOSEST neighbors  

Step 4: Make prediction based on neighbors  


### Distance Metrics

#### For Continuous Variables

**1. Euclidean Distance** (Most common)
- Creates straight-line ("as the crow flies") distance
- Formula: $d = \sqrt{\sum(x_i - y_i)^2}$
- Use when: Features are on similar scales

**2. Manhattan Distance**
- Follows grid-like paths (only horizontal/vertical moves)
- Formula: $d = \sum|x_i - y_i|$
- Use when: Movement is constrained (e.g., city blocks)

#### For Categorical Variables

**3. Hamming Distance**
- Counts number of positions where values differ
- Use for: Discrete categorical features

### Prediction Methods

#### 📊 kNN for Classification

**Method:** Use MODE (most common class) among k neighbors

**Example:** k=5 neighbors


Neighbor 1: Class A  
Neighbor 2: Class A  
Neighbor 3: Class B  
Neighbor 4: Class A  
Neighbor 5: Class B  

Prediction: Class A ✓  
(appears 3 times—the mode)  


#### 📈 kNN for Regression

**Method:** Use MEAN (average value) of k neighbors

**Example:** k=5 neighbors predicting units sold


Neighbor 1: 100 units  
Neighbor 2: 120 units  
Neighbor 3: 110 units  
Neighbor 4: 130 units  
Neighbor 5: 90 units  

Prediction: (100+120+110+130+90)/5 = 110 units ✓


### Choosing the Optimal k-Value

#### Systematic Approach


1. Start with k = √n  
   (where n = number of training samples)  
   Round to nearest odd number (avoids ties)  
   ↓  
2. Test range of k values  
   (k-4, k-2, k, k+2, k+4)  
   ↓  
3. Use cross-validation to evaluate each k  
   (typically 5-fold or 10-fold CV)  
   ↓  
4. Verify best k on test set  
   ↓  
5. Use optimal k for final model ✓  


#### Considerations

- **Small k (e.g., k=1):** 
  - More sensitive to noise
  - Higher variance, lower bias
  - Risk of **overfitting**
  
- **Large k (e.g., k=100):**
  - Smoother decision boundaries
  - Lower variance, higher bias
  - Risk of **underfitting**
  
- **Odd k:** Preferred for binary classification to avoid ties

**Best Practice:** Always use cross-validation rather than guessing k value

### kNN Implementation Example

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

"""
Assume:
'y' variable: Target variable/column (Units_Sold) we want to predict
'X' variable: All columns data that excluding our target variable (These columns are there to assist model to learn to predict 'y')
"""

# Scale features (IMPORTANT for kNN!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Find optimal k using cross-validation
k_values = range(1, 31, 2)  # Test odd values from 1 to 29
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_scaled, y, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Select best k
optimal_k = k_values[np.argmax(cv_scores)]
print(f"Optimal k: {optimal_k}")

# Train final model
final_knn = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn.fit(X_scaled, y)

### kNN Pros & Cons

#### Advantages ✅
- **Simple & Intuitive:** Easy to understand and implement
- **No Training Phase:** Instant "training" (just stores data)
- **Non-parametric:** Makes no assumptions about data distribution
- **Versatile:** Works for both classification and regression

#### Disadvantages ❌
- **Computationally Expensive:** Must calculate distance to all points for each prediction
- **Memory Intensive:** Stores entire training dataset
- **Curse of Dimensionality:** Performance degrades with many features
- **Requires Feature Scaling:** Sensitive to feature magnitudes
- **Sensitive to Irrelevant Features:** Distance calculations affected by noise

#### When to Use kNN
- Small to medium datasets
- Low-dimensional problems (< 20 features)
- Non-linear decision boundaries
- Quick prototyping and baseline model

---

## Summary: Key Takeaways

1. **ML Workflow:** Iterative cycle from business understanding to deployment
2. **EDA:** Systematic 4-step approach (head, describe numeric, describe categorical, clean)
3. **Missing Values:** Forward fill first, then segment median imputation
4. **Outliers:** Not always bad—consider domain knowledge before treating
5. **Feature Engineering:** Create informative features from raw data
6. **Feature Selection:** Remove redundant/irrelevant features to prevent overfitting
7. **Evaluation Metrics:** Choose based on business objective (MAE vs RMSE, Precision vs Recall)
8. **kNN:** Simple, instance-based algorithm—requires feature scaling and optimal k selection

---